In [1]:
import keras
import datetime
import tensorflow                       as tf
from tensorflow.keras.callbacks         import TensorBoard
from tensorflow.keras.layers            import Input, Lambda,UpSampling2D, Conv2D,Dropout,MaxPooling2D,Conv2DTranspose,concatenate,BatchNormalization, Activation
from tensorflow.keras.models            import Model
from tensorflow.keras.optimizers        import Adam,RMSprop
from keras.utils                        import plot_model
from tensorflow.keras                   import layers, models
from tensorflow.keras.losses            import mae
import sys
import os
import numpy as np
import math
import random, time
from pathlib                        import Path
from PIL                            import Image

import skimage                      as ski
from   skimage.filters              import threshold_otsu
from   skimage                      import io, color
from   skimage.color                import rgb2gray
from   skimage                      import filters
import cv2                          as cv
import matplotlib.pyplot            as plt 
import gc
import glob
from skimage                        import img_as_ubyte
from skimage                        import io
import shutil
tf.keras.backend.clear_session()

2025-06-15 23:04:15.516488: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-15 23:04:15.619963: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-15 23:04:15.687555: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-15 23:04:15.710715: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-15 23:04:15.793668: I tensorflow/core/platform/cpu_feature_guar

In [2]:
#general_path_data = '/home/ppgi/Trabajo/Codigos_generate_data/DLCODES-VER-5'
general_path_data ='/home/guiomar/Desktop/CODES/Generation_Data/Proyecto_UNI'

subdirectories = ['/Geometry','/Magnitude','/Pression','/U001','/U002']

geo_path =  general_path_data + subdirectories[0]
v_path   =  general_path_data + subdirectories[1]
p_path   =  general_path_data + subdirectories[2]
vx_path  =  general_path_data + subdirectories[3]
vy_path  =  general_path_data + subdirectories[4]

In [3]:
### method for reading image    
def get_img(img_name):
    return ski.io.imread(img_name)

###   method for resizing image
###   64 x 256
def resizing_img(x,new_width=256):
   
    height=x.shape[0]
    width=x.shape[1]
    ratio = height / width
    new_height = int(new_width * ratio)
    y =cv.resize(x,(new_width,new_height))
    return y

### method for turning to a grey or binary image 
def processing(image,option):
        x = get_img(image)  
        x = rgb2gray(x)               # It returns a grayscale image with floating point values in the range from 0 to 1
        y = resizing_img(x) # Reshape image 
    
        # Binary option otherwise gray
        if (option):
            y=ski.util.img_as_ubyte(y)  # Convert it back to the original data type and the data range back 0 to 255. 
                                        # It is often better to use image values represented by floating point values 
            best_value_threshold=np.round(filters.threshold_otsu(y)) #  Otsu’s method calculates an “optimal” threshold

            _,y= cv.threshold(y, best_value_threshold, 255, cv.THRESH_BINARY)
            y=y/255.
        y=y[:, :, np.newaxis]
        return y

def create_file(path,name='Images_masked'):
     newpath = path +'/' +name
     os.makedirs(newpath, exist_ok=True)
     return newpath

###  Method to create  npy object and save in a disk
def create_npy(path_origin,name_sub_file,option,n_sample = 1000):
    images = sorted([os.path.join(path_origin,fname) for fname in os.listdir(path_origin) if fname.endswith(".png")])
    current_path = os.getcwd() 
    if (n_sample <= 20000):

        I = images[1:n_sample+1]
        newpath1=create_file(current_path)
        new_root = newpath1 +'/' +name_sub_file
        os.makedirs(new_root, exist_ok=True)
   
        for img in I:
            name_img = os.path.splitext(os.path.basename(img))[0]
            save_in=new_root+'/'+ name_img+'.npy'
            print(save_in)
            x=processing(img,option)
            np.save(save_in,x)

### Method to apply mask, the mask is done by multiplication of two arrays
def mask(path1,path2,save_in,new_npy):
    
    path_1 = Path(path1)
    file_npy_1=sorted(list(path_1.glob('*.npy')))
    
    path_2 = Path(path2)
    file_npy_2=sorted(list(path_2.glob('*.npy')))
    n=1

    for f1, f2 in zip(file_npy_1,file_npy_2):
            new_save_in=save_in+ '/'+new_npy+f'_{n:05d}.npy'
            print(new_save_in)
            array_1 = np.load(f1)
            array_2 = np.load(f2)
            array_masked=array_1*array_2
            np.save(new_save_in,array_masked)
            print('*********',new_save_in)
            n+=1

def create_file_masked(file1,file2,name_npy,name):
    path1 =  os.getcwd() 
    path2= create_file(path1,name)
    mask(file1,file2,path2,name_npy)  

In [4]:
### Method for divided data in train,test and validation
def split_data(file,n_train,n_valid,n_test):
     n_sample=len(file)
     if (round(n_train + n_valid + n_test,2) == 1.0):
        n_train= math.floor(n_sample*n_train)
        n_valid = math.floor(n_sample*n_valid) + n_train
        n_test = math.floor(n_sample*n_test) + n_valid
        (train,valid,test) = (file[0:n_train], file[n_train:n_valid],file[n_valid:n_test]) 
     return train,valid,test

def create_files(ntrain,ntest,nval):
    
    name_files=['/G_Masked','/P_Masked','/V_Masked','/Vx_Masked','/Vy_Masked']
    current_path = os.getcwd()
    paths=[current_path+name_files[0],current_path+name_files[1],current_path+name_files[2],current_path+name_files[3],current_path+name_files[4]]
    for i, path in enumerate(paths):

        file = Path(path)
        arrays= sorted(file.glob('*.npy')) 
        train,valid,test =  split_data(arrays,ntrain,ntest,nval)

        path_train = paths[i]+'/train'
        path_test =  paths[i]+'/test'
        path_valid = paths[i]+'/valid'

        os.makedirs(path_train, exist_ok=True)
        os.makedirs(path_test, exist_ok=True)
        os.makedirs(path_valid, exist_ok=True)

        for array in train:
            p =  path_train + '/' + array.name
            shutil.move(array,p)

        for array in valid:
            p =  path_valid + '/' + array.name
            shutil.move(array,p)
        
        for array in test:
            p =  path_test+ '/' + array.name
            shutil.move(array,p)
        

In [ ]:
sample=20000

create_npy(geo_path,'Geo',True,n_sample = sample)
create_npy(v_path ,'V',False,n_sample = sample)
create_npy(p_path,'P',False,n_sample = sample)
create_npy(vx_path ,'Vx',False,n_sample = sample)
create_npy(vy_path ,'Vy',False,n_sample = sample)

In [ ]:
#path = '/home/ppgi/Trabajo/predicting-flow-patterns/'
path = '/home/guiomar/Desktop/CODES/predicting-flow-patterns/'
path1=path+'Images_masked/Geo'
path2=path+'Images_masked/P'
path3=path+'Images_masked/V'
path4=path+'Images_masked/Vx'
path5=path+'Images_masked/Vy'

list_paths = [path1,path2,path3,path4,path5]
labels = [('p','P_Masked'),('v','V_Masked'),('vx','Vx_Masked'),('vy','Vy_Masked')]

for k in range(len(labels)):
    create_file_masked(list_paths[0],list_paths[k+1],labels[k][0],labels[k][1])

In [ ]:
ntrain = 0.7
ntest  = 0.2
nval   = 0.1

create_files(ntrain,ntest,nval)

In [46]:


class MultiOutputNPYDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, x_paths, y_p_paths, y_v_paths,y_vx_paths, y_vy_paths, batch_size=10, shuffle=False):
        self.x_paths = x_paths
        self.y_p_paths = y_p_paths
        self.y_v_paths = y_v_paths
        self.y_vx_paths = y_vx_paths
        self.y_vy_paths = y_vy_paths
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.x_paths))

    def __len__(self):
        return int(np.ceil(len(self.x_paths) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index*self.batch_size : (index+1)*self.batch_size]
        
        batch_x = [self.x_paths[k] for k in batch_indexes]
        batch_p = [self.y_p_paths[k] for k in batch_indexes]
        batch_v = [self.y_v_paths[k] for k in batch_indexes]
        batch_vx = [self.y_vx_paths[k] for k in batch_indexes]
        batch_vy = [self.y_vy_paths[k] for k in batch_indexes]

        X = np.array([np.load(f) for f in batch_x])
        Y_p= np.array([np.load(f) for f in batch_p])
        Y_v = np.array([np.load(f) for f in batch_v])
        Y_vx = np.array([np.load(f) for f in batch_vx])
        Y_vy = np.array([np.load(f) for f in batch_vy])

        return X, [Y_p, Y_v, Y_vx,Y_vy]

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)




In [47]:
directory = '/home/guiomar/Desktop/CODES/predicting-flow-patterns'
#directory = '/home/ppgi/Trabajo/predicting-flow-patterns'

d1=directory+'/G_Masked'
d2=directory+'/P_Masked'
d3=directory+'/V_Masked'
d4=directory+'/Vx_Masked'
d5=directory+'/Vy_Masked'

gtrain= d1+'/train'
gtest=d1+'/test'
gval=d1+'/valid'

ptrain=d2+'/train'
ptest=d2+'/test'
pval=d2+'/valid'

vtrain=d3+'/train'
vtest=d3+'/test'
vval=d3+'/valid'

vxtrain=d4+'/train'
vxtest=d4+'/test'
vxval=d4+'/valid'

vytrain=d5+'/train'
vytest=d5+'/test'
vyval=d5+'/valid'

In [48]:
g_train = sorted(glob.glob(os.path.join(gtrain, "*.npy")))
g_test  = sorted(glob.glob(os.path.join(gtest, "*.npy")))
g_val   = sorted(glob.glob(os.path.join(gval, "*.npy")))

In [49]:
p_train = sorted(glob.glob(os.path.join(ptrain, "*.npy")))
p_test  = sorted(glob.glob(os.path.join(ptest, "*.npy")))
p_val   = sorted(glob.glob(os.path.join(pval, "*.npy")))

In [50]:
v_train = sorted(glob.glob(os.path.join(vtrain, "*.npy")))
v_test  = sorted(glob.glob(os.path.join(vtest, "*.npy")))
v_val   = sorted(glob.glob(os.path.join(vval, "*.npy")))

In [51]:
vx_train = sorted(glob.glob(os.path.join(vxtrain, "*.npy")))
vx_test  = sorted(glob.glob(os.path.join(vxtest, "*.npy")))
vx_val   = sorted(glob.glob(os.path.join(vxval, "*.npy")))

In [52]:
vy_train = sorted(glob.glob(os.path.join(vytrain, "*.npy")))
vy_test  = sorted(glob.glob(os.path.join(vytest, "*.npy")))
vy_val   = sorted(glob.glob(os.path.join(vyval, "*.npy")))

In [53]:

train_ds = MultiOutputNPYDataGenerator(g_train,p_train,v_train,vx_train,vy_train, batch_size=10, shuffle=False)
test_ds = MultiOutputNPYDataGenerator(g_test,p_test,v_test,vx_test,vy_test, batch_size=10, shuffle=False)
val_ds = MultiOutputNPYDataGenerator(g_val,p_val,v_val,vx_val,vy_val, batch_size=10, shuffle=False)


In [61]:
type(test_ds)


__main__.MultiOutputNPYDataGenerator

In [60]:
x_batch, y_batch = test_ds[201]  # primer bat


In [ ]:
for k,(batch_x, batch_y) in enumerate(test_ds):
    print(k)

TypeError: expected str, bytes or os.PathLike object, not ndarray